In [1]:
#!/usr/bin/env python3
"""
ENCODING — NEQR + Code 1 + QPA + EBA
======================================
Two operating modes controlled by PATCH_MODE flag:

  PATCH_MODE = False  (default / quantum verification mode)
    Processes a single 32×32 image as one quantum circuit.
    Full NEQR entanglement across all 1024 pixels simultaneously.
    Use this for QPA/EBA metrics, statistical analysis, paper results.

  PATCH_MODE = True  (visual demonstration mode)
    Divides a larger image (e.g. 128×128) into non-overlapping 32×32 patches.
    Each patch is processed by an independent full-NEQR circuit.
    Intra-patch entanglement is complete; inter-patch entanglement does not exist.
    Use this ONLY to produce presentable visual figures.

Outputs saved to OUT_DIR (neqr_shares_eba/).

Authentication:
  QPA — Quantum Parity Authentication   (8 CNOT / share, 24 total)
  EBA — Entanglement-Based Auth         (H + 2×CNOT / share, 9 total)
  Combined overhead: 9 qubits, 33 gates  (<0.02% of total gates)

Share logic (Code 1, nibble-level XOR):
  S1[LSB4] = G1[LSB4] XOR G2[LSB4]
  S1[MSB4] = C1[MSB4]
  S2[LSB4] = S1[MSB4] XOR G1[MSB4]
  S2[MSB4] = C2[MSB4]
  S3[LSB4] = S1[MSB4] XOR S2[MSB4] XOR G2[MSB4]
  S3[MSB4] = C3[MSB4]
"""


'\nENCODING — NEQR + Code 1 + QPA + EBA\n======================================\nTwo operating modes controlled by PATCH_MODE flag:\n\n  PATCH_MODE = False  (default / quantum verification mode)\n    Processes a single 32×32 image as one quantum circuit.\n    Full NEQR entanglement across all 1024 pixels simultaneously.\n    Use this for QPA/EBA metrics, statistical analysis, paper results.\n\n  PATCH_MODE = True  (visual demonstration mode)\n    Divides a larger image (e.g. 128×128) into non-overlapping 32×32 patches.\n    Each patch is processed by an independent full-NEQR circuit.\n    Intra-patch entanglement is complete; inter-patch entanglement does not exist.\n    Use this ONLY to produce presentable visual figures.\n\nOutputs saved to OUT_DIR (neqr_shares_eba/).\n\nAuthentication:\n  QPA — Quantum Parity Authentication   (8 CNOT / share, 24 total)\n  EBA — Entanglement-Based Auth         (H + 2×CNOT / share, 9 total)\n  Combined overhead: 9 qubits, 33 gates  (<0.02% of total ga

In [2]:
import os, sys, json, math, time
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from mqt.ddsim import DDSIMProvider

# ── Import shared utilities ───────────────────────────────────────────
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from qvss_utils import (
    to_bits, classical_parity, image_entropy, calc_metrics,
    get_npos, encode_neqr, add_qpa, add_eba,
    build_encoding_circuit, parse_encoding_results,
    extract_secret_nibble,
)

# ═════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit here
# ═════════════════════════════════════════════════════════════════════
PATCH_MODE  = True        # False = full 32×32 quantum | True = patch-based visual
PATCH_SIZE  = (16, 16)     # patch dimensions (always 32×32)
FULL_IMG_SIZE = (256, 256) # used only when PATCH_MODE = True

N_SECRETS   = 2
BITS        = 8
SHOTS       = 8192 
OUT_DIR     = 'neqr_shares_eba'   # ← single canonical output dir

# Image sets for multi-image analysis (PATCH_MODE=False only).
# Each entry: (secret1_path, secret2_path, cover1_path, cover2_path, cover3_path, label)
IMAGE_SETS = [
    ('secret_1', 'secret_2', 'cover_1', 'cover_2', 'cover_3', 'set_A'),
]

provider    = DDSIMProvider()
SIM_BACKEND = provider.get_backend('qasm_simulator')
os.makedirs(OUT_DIR, exist_ok=True)
print("[Config]")
print(f"  PATCH_MODE={PATCH_MODE}  PATCH_SIZE={PATCH_SIZE}")
print(f"  FULL_IMG_SIZE={FULL_IMG_SIZE}  SHOTS={SHOTS}  OUT_DIR={OUT_DIR}")


[Config]
  PATCH_MODE=True  PATCH_SIZE=(16, 16)
  FULL_IMG_SIZE=(256, 256)  SHOTS=8192  OUT_DIR=neqr_shares_eba


In [3]:
# ── Image loading ─────────────────────────────────────────────────────
def load_or_generate(base_path, rows, cols, demo_type='random'):
    """Load image resized to rows×cols, or generate random demo."""
    for ext in ['.png', '.jpg', '.jpeg', '.tif']:
        p = os.path.splitext(base_path)[0] + ext
        if os.path.exists(p):
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                resized = cv2.resize(img, (cols, rows), interpolation=cv2.INTER_AREA)
                print(f"  Loaded: {os.path.basename(p)} → resized to {rows}×{cols}")
                return resized
    print(f"  Generating random demo ({demo_type}) for {base_path}")
    seed = sum(ord(c) for c in base_path) % 1000
    np.random.seed(seed)
    img = np.random.randint(40, 215, (rows, cols), np.uint8)
    cv2.imwrite(base_path + '.png', img)
    return img


def verify_first_pixel(secrets, covers, shares, rows, cols):
    """Classical ground-truth check of share values at pixel (0,0)."""
    G1,G2 = int(secrets[0].flatten()[0]), int(secrets[1].flatten()[0])
    C1,C2,C3 = int(covers[0].flatten()[0]), int(covers[1].flatten()[0]), int(covers[2].flatten()[0])
    S1_exp = ((G1 & 0x0F) ^ (G2 & 0x0F)) | (C1 & 0xF0)
    S2_lsb = ((S1_exp >> 4) ^ (G1 >> 4)) & 0x0F
    S2_exp = S2_lsb | (C2 & 0xF0)
    S3_lsb = ((S1_exp >> 4) ^ (S2_exp >> 4) ^ (G2 >> 4)) & 0x0F
    S3_exp = S3_lsb | (C3 & 0xF0)
    print("=" * 60)
    print("SHARE VERIFICATION — pixel (0,0)")
    print("=" * 60)
    print(f"  G1={G1}  G2={G2}  C1={C1}  C2={C2}  C3={C3}")
    for name, got, exp in [('S1', int(shares['S1'][0,0]), S1_exp),
                            ('S2', int(shares['S2'][0,0]), S2_exp),
                            ('S3', int(shares['S3'][0,0]), S3_exp)]:
        ok = '✓ OK' if got == exp else '✗ FAIL'
        print(f"  {name}: got={got:3d}  expected={exp:3d}  {ok}")


def print_auth_validation(qpa_stats, eba_stats):
    """Print QPA and EBA accuracy summary."""
    print("\n" + "=" * 60)
    print("AUTHENTICATION VALIDATION")
    print("=" * 60)
    print("QPA (Quantum Parity Authentication):")
    for sname in ['S1','S2','S3']:
        st = qpa_stats[sname]
        total = st['correct'] + st['wrong']
        acc = 100 * st['correct'] / total if total else 0
        print(f"  {sname}: {st['correct']}/{total}  acc={acc:.2f}%  "
              f"{'PASS' if acc >= 99 else 'FAIL'}")
    print("EBA (Entanglement-Based Authentication):")
    for sname in ['S1','S2','S3']:
        st = eba_stats[sname]
        total = st['correct'] + st['wrong']
        acc = 100 * st['correct'] / total if total else 0
        print(f"  {sname}: {st['correct']}/{total}  acc={acc:.2f}%  "
              f"{'PASS' if acc >= 99 else 'FAIL'}")
    qpa_ok = all(
        100 * qpa_stats[s]['correct'] / max(1, qpa_stats[s]['correct'] + qpa_stats[s]['wrong']) >= 99
        for s in ['S1','S2','S3'])
    eba_ok = all(
        100 * eba_stats[s]['correct'] / max(1, eba_stats[s]['correct'] + eba_stats[s]['wrong']) >= 99
        for s in ['S1','S2','S3'])
    print(f"\n  QPA overall: {'PASS' if qpa_ok else 'FAIL'}")
    print(f"  EBA overall: {'PASS' if eba_ok else 'FAIL'}")
    return qpa_ok, eba_ok


In [4]:
# ── EBA proof plot ────────────────────────────────────────────────────
def save_eba_plot(eba_stats, qpa_stats, tag='encoding', out_dir=OUT_DIR):
    fig = plt.figure(figsize=(16, 8))
    fig.patch.set_facecolor('#0d1117')
    gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)
    BG='#161b22'; TEXT='#e6edf3'; GRID='#30363d'
    GRN='#00e676'; RED='#ff1744'; GOLD='#ffd600'; BLUE='#448aff'

    def sax(ax, title):
        ax.set_facecolor(BG); ax.tick_params(colors=TEXT, labelsize=9)
        ax.title.set_color(TEXT); ax.title.set_fontsize(11); ax.title.set_fontweight('bold')
        for sp in ax.spines.values(): sp.set_edgecolor(GRID)
        ax.set_title(title); ax.grid(color=GRID, linestyle='--', alpha=0.5)

    for idx, sname in enumerate(['S1','S2','S3']):
        ax = fig.add_subplot(gs[0, idx]); sax(ax, f'{sname} EBA Correlation')
        st = eba_stats[sname]; total = st['correct'] + st['wrong']
        acc = 100 * st['correct'] / total if total else 0
        bars = ax.bar(['Correlated', 'Violated'], [acc, 100-acc], color=[GRN, RED], edgecolor=GRID)
        ax.axhline(99, color=GOLD, linestyle='--', linewidth=1.5, label='99% threshold')
        ax.set_ylim(0, 115); ax.set_ylabel('%', color=TEXT)
        ax.legend(fontsize=8, labelcolor=TEXT, facecolor=BG)
        for bar, v in zip(bars, [acc, 100-acc]):
            ax.text(bar.get_x()+bar.get_width()/2, v+1.5, f'{v:.2f}%',
                    ha='center', color=TEXT, fontsize=11, fontweight='bold')

    ax2 = fig.add_subplot(gs[1, :]); sax(ax2, 'EBA vs QPA Per-Shot Accuracy')
    x = np.arange(3); w = 0.3
    eba_acc = [100*eba_stats[s]['correct']/max(1,eba_stats[s]['correct']+eba_stats[s]['wrong']) for s in ['S1','S2','S3']]
    qpa_acc = [100*qpa_stats[s]['correct']/max(1,qpa_stats[s]['correct']+qpa_stats[s]['wrong']) for s in ['S1','S2','S3']]
    b1 = ax2.bar(x-w/2, eba_acc, w, label='EBA (Bell corr.)', color=BLUE, edgecolor=GRID)
    b2 = ax2.bar(x+w/2, qpa_acc, w, label='QPA (Parity)',    color=GRN,  edgecolor=GRID)
    ax2.axhline(99, color=GOLD, linestyle='--', linewidth=1.2, label='99% threshold')
    ax2.set_xticks(x); ax2.set_xticklabels(['S1','S2','S3'])
    ax2.set_ylim(85, 105); ax2.set_ylabel('Accuracy %', color=TEXT)
    ax2.legend(fontsize=9, labelcolor=TEXT, facecolor=BG)
    for bar, v in list(zip(b1, eba_acc)) + list(zip(b2, qpa_acc)):
        ax2.text(bar.get_x()+bar.get_width()/2, v+0.2, f'{v:.1f}%',
                 ha='center', color=TEXT, fontsize=9, fontweight='bold')

    fig.suptitle(f'EBA + QPA Authentication ({tag.upper()})\n'
                 f'Bell |Φ+⟩: MSB==0→anc==tag, MSB==1→anc≠tag',
                 color=TEXT, fontsize=13, fontweight='bold', y=0.99)
    out = os.path.join(out_dir, f'eba_qpa_proof_{tag}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.close(); print(f"  Plot saved → {out}")
    return out


In [5]:
# ── Full-mode runner (32×32, single image set) ────────────────────────
def run_full_mode(secrets, covers, rows, cols, label='A', out_dir=OUT_DIR):
    """
    Encode one set of secrets+covers using full NEQR (no patching).
    rows × cols must equal PATCH_SIZE (32×32).
    """
    npy, npx, npos = get_npos(rows, cols)
    print(f"\n[Full-mode encoding]  size={rows}×{cols}  label={label}")
    print(f"  NPOS={npos}  TotalQ={npos+5*8+3*8+3+6}")

    t0 = time.time()
    qc = build_encoding_circuit(secrets, covers, rows, cols)
    build_t = time.time() - t0
    print(f"  Circuit built: {qc.num_qubits} qubits  depth={qc.depth()}  "
          f"gates={len(qc.data)}  ({build_t:.1f}s)")

    t1 = time.time()
    counts  = SIM_BACKEND.run(qc, shots=SHOTS).result().get_counts()
    sim_t   = time.time() - t1
    print(f"  Simulation: {sim_t:.1f}s  unique_outcomes={len(counts)}")

    shares, parity_maps, qpa_stats, eba_stats = parse_encoding_results(counts, rows, cols)
    verify_first_pixel(secrets, covers, shares, rows, cols)
    qpa_ok, eba_ok = print_auth_validation(qpa_stats, eba_stats)

    # QPA parity map vs classical ground truth
    print("\n[QPA parity map vs classical ground truth]")
    for sname, img in shares.items():
        pmap  = parity_maps[sname]
        valid = pmap >= 0
        if valid.any():
            expected = np.vectorize(classical_parity)(img)
            match    = np.sum(pmap[valid] == expected[valid])
            total_v  = int(valid.sum())
            print(f"  {sname}: {match}/{total_v} "
                  f"({100*match/total_v:.1f}%)  coverage={100*total_v/(rows*cols):.1f}%")

    # Save outputs
    set_dir = os.path.join(out_dir, f'set_{label}')
    os.makedirs(set_dir, exist_ok=True)
    for name, img in shares.items():
        cv2.imwrite(os.path.join(set_dir, f'{name}.png'), img)
        np.save(os.path.join(set_dir, f'{name}.npy'), img)
    for i, img in enumerate(secrets):
        cv2.imwrite(os.path.join(set_dir, f'G{i+1}_original.png'), img)
    for i, img in enumerate(covers):
        cv2.imwrite(os.path.join(set_dir, f'C{i+1}_original.png'), img)
    for name, pmap in parity_maps.items():
        np.save(os.path.join(set_dir, f'parity_map_{name}.npy'), pmap)
    for name, img in shares.items():
        eba_map = ((img >> 7) & 1).astype(np.uint8)
        np.save(os.path.join(set_dir, f'eba_map_{name}.npy'), eba_map)

    meta = {
        'label': label, 'mode': 'full',
        'rows': rows, 'cols': cols, 'shots': SHOTS,
        'total_qubits': qc.num_qubits, 'depth': qc.depth(),
        'gates': len(qc.data), 'build_time': build_t, 'sim_time': sim_t,
        'qpa_ok': qpa_ok, 'eba_ok': eba_ok,
        'qpa_stats': qpa_stats, 'eba_stats': eba_stats,
    }
    with open(os.path.join(set_dir, 'encoding_meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    save_eba_plot(eba_stats, qpa_stats, tag=f'encoding_{label}', out_dir=set_dir)

    for sname, img in shares.items():
        hist, _ = np.histogram(img, bins=256, range=(0,255))
        p   = hist / img.size
        ent = float(-np.sum(p * np.log2(p + 1e-10)))
        print(f"  {sname}: entropy={ent:.4f} bits  "
              f"min={img.min()}  max={img.max()}  mean={img.mean():.1f}")

    print(f"\n  Saved → {set_dir}/")
    print(f"  Total time: {build_t+sim_t:.1f}s")
    return shares, parity_maps, qpa_stats, eba_stats, meta


In [6]:
# ── Patch-mode runner ─────────────────────────────────────────────────
def run_patch_mode(all_secrets, all_covers, img_h, img_w,
                   ph=32, pw=32, out_dir=OUT_DIR, label='patch'):
    """
    Process a larger image (img_h × img_w) by dividing into ph×pw patches.
    Each patch is encoded with a full independent NEQR+QPA+EBA circuit.
    Intra-patch entanglement is complete; inter-patch entanglement does not exist.
    This mode is for visual demonstration only — do not use for statistical analysis.
    """
    assert img_h % ph == 0 and img_w % pw == 0,         f"Image ({img_h}×{img_w}) not divisible by patch ({ph}×{pw})"
    ny, nx     = img_h // ph, img_w // pw
    n_patches  = ny * nx

    print(f"\n[Patch-mode encoding]  image={img_h}×{img_w}  "
          f"patch={ph}×{pw}  patches={n_patches}")

    full_shares  = {k: np.zeros((img_h, img_w), np.uint8) for k in ['S1','S2','S3']}
    full_parity  = {k: -np.ones((img_h, img_w), int)      for k in ['S1','S2','S3']}
    acc_qpa      = {k: {'correct': 0, 'wrong': 0} for k in ['S1','S2','S3']}
    acc_eba      = {k: {'correct': 0, 'wrong': 0} for k in ['S1','S2','S3']}
    t_total      = 0.0
    idx          = 0

    for py in range(ny):
        for px in range(nx):
            idx += 1
            y0, y1 = py * ph, (py+1) * ph
            x0, x1 = px * pw, (px+1) * pw
            print(f"  Patch {idx:3d}/{n_patches}  rows=[{y0}:{y1}] cols=[{x0}:{x1}]",
                  end='  ', flush=True)

            secret_p = [s[y0:y1, x0:x1] for s in all_secrets]
            cover_p  = [c[y0:y1, x0:x1] for c in all_covers]

            t0 = time.time()
            qc = build_encoding_circuit(secret_p, cover_p, ph, pw)
            counts = SIM_BACKEND.run(qc, shots=SHOTS).result().get_counts()
            elapsed = time.time() - t0
            t_total += elapsed

            p_shares, p_parity, p_qpa, p_eba = parse_encoding_results(counts, ph, pw)
            print(f"{elapsed:.1f}s")

            for k in ['S1','S2','S3']:
                full_shares[k][y0:y1, x0:x1] = p_shares[k]
                full_parity[k][y0:y1, x0:x1] = p_parity[k]
                acc_qpa[k]['correct'] += p_qpa[k]['correct']
                acc_qpa[k]['wrong']   += p_qpa[k]['wrong']
                acc_eba[k]['correct'] += p_eba[k]['correct']
                acc_eba[k]['wrong']   += p_eba[k]['wrong']

    print(f"\n  All patches done.  Total time: {t_total:.1f}s ({t_total/60:.1f} min)")

    # Save
    set_dir = os.path.join(out_dir, f'patch_{img_h}x{img_w}_{label}')
    os.makedirs(set_dir, exist_ok=True)
    for name, img in full_shares.items():
        cv2.imwrite(os.path.join(set_dir, f'{name}.png'), img)
        np.save(os.path.join(set_dir, f'{name}.npy'), img)
        eba_map = ((img >> 7) & 1).astype(np.uint8)
        np.save(os.path.join(set_dir, f'eba_map_{name}.npy'), eba_map)
    for name, pmap in full_parity.items():
        np.save(os.path.join(set_dir, f'parity_map_{name}.npy'), pmap)
    for i, img in enumerate(all_secrets):
        cv2.imwrite(os.path.join(set_dir, f'G{i+1}_original.png'), img)
    for i, img in enumerate(all_covers):
        cv2.imwrite(os.path.join(set_dir, f'C{i+1}_original.png'), img)

    meta = {
        'mode': 'patch', 'img_size': f'{img_h}x{img_w}',
        'patch_size': f'{ph}x{pw}', 'n_patches': n_patches,
        'shots': SHOTS, 'total_time': t_total,
        'qpa_stats_agg': acc_qpa, 'eba_stats_agg': acc_eba,
        'note': 'Patch mode: intra-patch entanglement only. Visual demo use only.',
    }
    with open(os.path.join(set_dir, 'encoding_meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    print_auth_validation(acc_qpa, acc_eba)
    print(f"  Saved → {set_dir}/")
    return full_shares, full_parity, acc_qpa, acc_eba


In [7]:
# ═══════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════
print("=" * 70)
print("QVSS ENCODING — NEQR + Code1 + QPA + EBA")
print("=" * 70)

ph, pw = PATCH_SIZE

if not PATCH_MODE:
    # ── Full quantum mode: one image set at a time ──────────────────
    print(f"\nMode: FULL NEQR  ({ph}×{pw} per image set)")
    for (s1_p, s2_p, c1_p, c2_p, c3_p, lbl) in IMAGE_SETS:
        print(f"\n── Image set: {lbl} ──")
        secrets = [load_or_generate(s1_p, ph, pw),
                   load_or_generate(s2_p, ph, pw)]
        covers  = [load_or_generate(c1_p, ph, pw),
                   load_or_generate(c2_p, ph, pw),
                   load_or_generate(c3_p, ph, pw)]
        run_full_mode(secrets, covers, ph, pw, label=lbl)

else:
    # ── Patch mode: one large image pair ───────────────────────────
    img_h, img_w = FULL_IMG_SIZE
    print(f"\nMode: PATCH  ({img_h}×{img_w} → {ph}×{pw} patches)")
    print("NOTE: Use for visual figures only, not for statistical analysis.")
    # Use first image set paths, loaded at full resolution
    s1_p, s2_p, c1_p, c2_p, c3_p, lbl = IMAGE_SETS[0]
    all_secrets = [load_or_generate(s1_p, img_h, img_w),
                   load_or_generate(s2_p, img_h, img_w)]
    all_covers  = [load_or_generate(c1_p, img_h, img_w),
                   load_or_generate(c2_p, img_h, img_w),
                   load_or_generate(c3_p, img_h, img_w)]
    run_patch_mode(all_secrets, all_covers, img_h, img_w,
                   ph=ph, pw=pw, label=lbl)

print("\nDONE. Run decoding_verification.ipynb next.")


QVSS ENCODING — NEQR + Code1 + QPA + EBA

Mode: PATCH  (256×256 → 16×16 patches)
NOTE: Use for visual figures only, not for statistical analysis.
  Loaded: secret_1.tif → resized to 256×256
  Loaded: secret_2.tif → resized to 256×256
  Loaded: cover_1.tif → resized to 256×256
  Loaded: cover_2.tif → resized to 256×256
  Loaded: cover_3.tif → resized to 256×256

[Patch-mode encoding]  image=256×256  patch=16×16  patches=256
  Patch   1/256  rows=[0:16] cols=[0:16]  7.8s
  Patch   2/256  rows=[0:16] cols=[16:32]  6.8s
  Patch   3/256  rows=[0:16] cols=[32:48]  6.6s
  Patch   4/256  rows=[0:16] cols=[48:64]  15.7s
  Patch   5/256  rows=[0:16] cols=[64:80]  17.5s
  Patch   6/256  rows=[0:16] cols=[80:96]  21.9s
  Patch   7/256  rows=[0:16] cols=[96:112]  22.2s
  Patch   8/256  rows=[0:16] cols=[112:128]  18.2s
  Patch   9/256  rows=[0:16] cols=[128:144]  12.2s
  Patch  10/256  rows=[0:16] cols=[144:160]  18.4s
  Patch  11/256  rows=[0:16] cols=[160:176]  24.4s
  Patch  12/256  rows=[0:16] 